In [ ]:
# ============================================================
# CELL 1 — Build graph cache with angle triplets
#
# What this cell does:
#   1) Read the full pair database CSV
#   2) Parse polar / nonpolar structure IDs
#   3) Check CIF availability
#   4) Build periodic graphs with:
#        - radius cutoff
#        - per-center topK neighbors
#        - forced bidirectional edges
#        - angle triplets on primary edges
#   5) Save per-structure graph cache (.joblib)
#   6) Save graph statistics and pair index tables
#
# Optional:
#   If BUILD_LOADERS=True and SPLIT_MAP_PATH is provided, also build
#   dl_tr / dl_va / dl_te for quick inspection.
# ============================================================

import os
import re
import glob
import time
import math
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import torch
from torch.utils.data import Dataset, DataLoader
from ase.io import read as ase_read
from ase.neighborlist import neighbor_list


# ============================================================
# 0) User config
# ============================================================
DATA_PATH = "./data/pair_database.csv"
CIF_DIR = "./data/structures"

OUT_DIR = "./artifacts/graph_cache_angle"
CACHE_DIR = os.path.join(OUT_DIR, "graph_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# Optional split map for quick dataloader construction
SPLIT_MAP_PATH = None   # e.g. "./data/split_map.csv"
BUILD_LOADERS = False

OUT_GRAPH_STATS = os.path.join(OUT_DIR, "graph_stats.csv")
OUT_MISSING_CIF = os.path.join(OUT_DIR, "missing_cif_rows.csv")
OUT_PAIR_INDEX = os.path.join(OUT_DIR, "pair_index_from_db.csv")


# ============================================================
# 1) CSV column names
# ============================================================
POL_PAIRKEY_COL = "Polar_mpid"
NPOL_ID_COL = "NPolar_mpid"
DE_COL = "Energy_diff_meV"

FORM_COL_CANDIDATES = ["Polar_pretty_formula", "pretty_formula", "formula"]


# ============================================================
# 2) Graph config
# ============================================================
R_MAX = 6.0
MAX_NEIGHBORS = 64
SELF_LOOPS = False

K_ANGLE = 12
MAX_TRIPLETS_PER_CENTER = 200

# Optional loaders
BATCH_SIZE = 8
NUM_WORKERS = 0
PIN_MEMORY = True


# ============================================================
# 3) Helpers: CIF index + strict ID parsing
# ============================================================
def build_cif_index(cif_dir: str):
    files = glob.glob(os.path.join(cif_dir, "*.cif"))
    if len(files) == 0:
        files = glob.glob(os.path.join(cif_dir, "**", "*.cif"), recursive=True)

    idx = {}
    dup = 0
    for fp in files:
        key = os.path.splitext(os.path.basename(fp))[0].strip().lower()
        if key in idx:
            dup += 1
            continue
        idx[key] = fp

    print(f"[CIF] indexed: {len(idx)} files (raw={len(files)}, dups_skipped={dup}) under: {cif_dir}")
    return idx


CIF_INDEX = build_cif_index(CIF_DIR)


def normalize_token(s: str) -> str:
    s = str(s).strip()
    s = os.path.basename(s)
    if s.lower().endswith(".cif"):
        s = s[:-4]
    return s.strip()


_ID_TOKEN_RE = re.compile(r"(?i)^[a-z]+-\d+$|^\d+$")
_ID_EXTRACT_RE = re.compile(r"(?i)([a-z]+-\d+|\d+)")


def extract_id_token(s: str) -> str:
    s = normalize_token(s)
    m = _ID_EXTRACT_RE.findall(s)
    if not m:
        raise ValueError(f"Cannot extract ID token from: {s}")
    return m[0]


def assert_struct_id(s: str) -> str:
    s = normalize_token(s)
    if "__" in s:
        raise ValueError(f"[BUG] struct_id contains '__' (looks like pairkey): {s}")
    tok = extract_id_token(s)
    if not _ID_TOKEN_RE.match(tok):
        raise ValueError(f"Bad struct id token: {tok} from {s}")
    return tok


def parse_pairkey(pairkey: str, fallback_npolar: str = None):
    s = normalize_token(pairkey)
    if "__" in s:
        a, b = s.split("__", 1)
        return extract_id_token(a), extract_id_token(b)
    if fallback_npolar is not None:
        return extract_id_token(s), extract_id_token(fallback_npolar)
    raise ValueError(f"Cannot parse pairkey: {pairkey}")


def resolve_cif_path(struct_id: str) -> str:
    sid = assert_struct_id(struct_id)
    key = sid.lower()
    if key in CIF_INDEX:
        return CIF_INDEX[key]

    # Optional fuzzy fallback
    cands = [k for k in CIF_INDEX.keys() if key in k]
    if not cands:
        raise FileNotFoundError(f"Cannot find CIF for id={sid} under {CIF_DIR}")
    cands = sorted(cands, key=lambda x: (len(x), x))
    return CIF_INDEX[cands[0]]


def cache_file(struct_id: str) -> str:
    sid = assert_struct_id(struct_id)
    return os.path.join(CACHE_DIR, f"{sid}.joblib")


def _unique_rows_int64(mat: np.ndarray):
    v = np.ascontiguousarray(mat).view(np.dtype((np.void, mat.dtype.itemsize * mat.shape[1])))
    _, first = np.unique(v, return_index=True)
    first = np.sort(first)
    return first


# ============================================================
# 4) Graph build: V2 + ANGLE
# ============================================================
def build_pbc_graph_v2_angle_with_remap(atoms, r_max: float, max_neighbors: int, self_loops: bool, k_angle: int):
    pos = atoms.get_positions().astype(np.float32)
    z = atoms.get_atomic_numbers().astype(np.int64)
    cell = atoms.get_cell().array.astype(np.float32)
    pbc = np.array(atoms.get_pbc(), dtype=np.bool_)

    i, j, S, _d = neighbor_list("ijSd", atoms, cutoff=r_max, self_interaction=self_loops)

    if len(i) == 0:
        return dict(
            z=z,
            pos=pos,
            cell=cell,
            pbc=pbc,
            edge_index=np.zeros((2, 0), dtype=np.int64),
            edge_vec=np.zeros((0, 3), dtype=np.float32),
            edge_dist=np.zeros((0,), dtype=np.float32),
            edge_shift=np.zeros((0, 3), dtype=np.float32),
            triplet_center=np.zeros((0,), dtype=np.int64),
            triplet_e1=np.zeros((0,), dtype=np.int64),
            triplet_e2=np.zeros((0,), dtype=np.int64),
            triplet_cos=np.zeros((0,), dtype=np.float32),
        )

    i = i.astype(np.int64)
    j = j.astype(np.int64)
    S = S.astype(np.int64)

    shift = (S.astype(np.float32) @ cell).astype(np.float32)
    rij = (pos[j] + shift - pos[i]).astype(np.float32)
    dist = np.linalg.norm(rij, axis=1).astype(np.float32)

    # Primary edges: per-center topK
    if max_neighbors is not None and max_neighbors > 0:
        order = np.lexsort((dist, i))
        i2, j2, rij2, dist2, S2 = i[order], j[order], rij[order], dist[order], S[order]

        keep = np.zeros(len(i2), dtype=bool)
        start = 0
        while start < len(i2):
            ii = i2[start]
            end = start + 1
            while end < len(i2) and i2[end] == ii:
                end += 1
            k = min(int(max_neighbors), end - start)
            keep[start:start + k] = True
            start = end

        i2, j2, rij2, dist2, S2 = i2[keep], j2[keep], rij2[keep], dist2[keep], S2[keep]
    else:
        i2, j2, rij2, dist2, S2 = i, j, rij, dist, S

    E0 = int(len(i2))

    # Angle triplets on primary edges
    trip_c, trip_e1, trip_e2, trip_cos = [], [], [], []
    if E0 >= 2:
        ord_i = np.argsort(i2, kind="mergesort")
        i_sorted = i2[ord_i]
        start = 0
        while start < E0:
            c = int(i_sorted[start])
            end = start + 1
            while end < E0 and int(i_sorted[end]) == c:
                end += 1

            eidx = ord_i[start:end]
            m = len(eidx)
            if m >= 2:
                if m > k_angle:
                    sel = np.argsort(dist2[eidx], kind="mergesort")[:k_angle]
                    eidx = eidx[sel]
                    m = len(eidx)

                if m >= 2:
                    U = rij2[eidx] / (dist2[eidx][:, None] + 1e-12)
                    cosmat = (U @ U.T).astype(np.float32)
                    t0, t1 = np.triu_indices(m, k=1)

                    if t0.size > 0:
                        cosv = cosmat[t0, t1]
                        e1 = eidx[t0].astype(np.int64)
                        e2 = eidx[t1].astype(np.int64)

                        if cosv.size > MAX_TRIPLETS_PER_CENTER:
                            cosv = cosv[:MAX_TRIPLETS_PER_CENTER]
                            e1 = e1[:MAX_TRIPLETS_PER_CENTER]
                            e2 = e2[:MAX_TRIPLETS_PER_CENTER]

                        trip_c.append(np.full((len(cosv),), c, dtype=np.int64))
                        trip_e1.append(e1)
                        trip_e2.append(e2)
                        trip_cos.append(cosv.astype(np.float32))

            start = end

    if len(trip_c) == 0:
        triplet_center0 = np.zeros((0,), dtype=np.int64)
        triplet_e10 = np.zeros((0,), dtype=np.int64)
        triplet_e20 = np.zeros((0,), dtype=np.int64)
        triplet_cos0 = np.zeros((0,), dtype=np.float32)
    else:
        triplet_center0 = np.concatenate(trip_c, axis=0)
        triplet_e10 = np.concatenate(trip_e1, axis=0)
        triplet_e20 = np.concatenate(trip_e2, axis=0)
        triplet_cos0 = np.concatenate(trip_cos, axis=0)

    # Force bidirectional edges
    i_all0 = np.concatenate([i2, j2], axis=0).astype(np.int64)
    j_all0 = np.concatenate([j2, i2], axis=0).astype(np.int64)
    S_all0 = np.concatenate([S2, -S2], axis=0).astype(np.int64)
    rij_all0 = np.concatenate([rij2, -rij2], axis=0).astype(np.float32)
    dist_all0 = np.concatenate([dist2, dist2], axis=0).astype(np.float32)

    # Deduplicate by (i, j, S)
    key0 = np.stack([i_all0, j_all0, S_all0[:, 0], S_all0[:, 1], S_all0[:, 2]], axis=1).astype(np.int64)
    first = _unique_rows_int64(key0)

    new_index = -np.ones((len(i_all0),), dtype=np.int64)
    new_index[first] = np.arange(len(first), dtype=np.int64)

    # Remap triplets
    if triplet_e10.size > 0:
        ne1 = new_index[triplet_e10]
        ne2 = new_index[triplet_e20]
        ok = (ne1 >= 0) & (ne2 >= 0)
        triplet_center = triplet_center0[ok]
        triplet_e1 = ne1[ok].astype(np.int64)
        triplet_e2 = ne2[ok].astype(np.int64)
        triplet_cos = triplet_cos0[ok].astype(np.float32)
    else:
        triplet_center = triplet_center0
        triplet_e1 = triplet_e10
        triplet_e2 = triplet_e20
        triplet_cos = triplet_cos0

    i_all = i_all0[first]
    j_all = j_all0[first]
    S_all = S_all0[first]
    rij_all = rij_all0[first]
    dist_all = dist_all0[first]
    shift_all = (S_all.astype(np.float32) @ cell).astype(np.float32)

    edge_index = np.stack([i_all, j_all], axis=0)

    return dict(
        z=z,
        pos=pos,
        cell=cell,
        pbc=pbc,
        edge_index=edge_index,
        edge_vec=rij_all,
        edge_dist=dist_all,
        edge_shift=shift_all,
        triplet_center=triplet_center,
        triplet_e1=triplet_e1,
        triplet_e2=triplet_e2,
        triplet_cos=triplet_cos,
    )


def load_or_build_graph(struct_id: str):
    sid = assert_struct_id(struct_id)
    cpath = cache_file(sid)
    if os.path.exists(cpath):
        return joblib.load(cpath)

    cif = resolve_cif_path(sid)
    atoms = ase_read(cif)
    g = build_pbc_graph_v2_angle_with_remap(
        atoms,
        r_max=R_MAX,
        max_neighbors=MAX_NEIGHBORS,
        self_loops=SELF_LOOPS,
        k_angle=K_ANGLE,
    )
    g["id"] = sid
    g["cif_path"] = cif
    g["natoms"] = int(len(g["z"]))
    joblib.dump(g, cpath)
    return g


# ============================================================
# 5) Read full database CSV + build ID arrays
# ============================================================
df = pd.read_csv(DATA_PATH)
print(f"[CSV] rows={len(df)} cols={len(df.columns)}")

for c in [POL_PAIRKEY_COL, NPOL_ID_COL, DE_COL]:
    if c not in df.columns:
        raise KeyError(f"Missing column '{c}' in DATA_PATH. Found: {list(df.columns)}")

y_de = pd.to_numeric(df[DE_COL], errors="coerce").values.astype(np.float32)
if not np.isfinite(y_de).all():
    bad = np.where(~np.isfinite(y_de))[0][:20]
    raise ValueError(f"dE has NaN/Inf rows={bad}")

form_col = next((c for c in FORM_COL_CANDIDATES if c in df.columns), None)

polar_id = np.empty(len(df), dtype=object)
npolar_id = np.empty(len(df), dtype=object)
mismatch = 0

for i, (pk, ncol) in enumerate(zip(df[POL_PAIRKEY_COL].values, df[NPOL_ID_COL].values)):
    p, n_from_pk = parse_pairkey(pk, fallback_npolar=ncol)
    n = extract_id_token(ncol)
    polar_id[i] = assert_struct_id(p)
    npolar_id[i] = assert_struct_id(n)
    if normalize_token(n_from_pk) != normalize_token(n):
        mismatch += 1

print(f"[SANITY] npolar mismatch between pairkey and {NPOL_ID_COL} = {mismatch}/{len(df)}")
print(f"[SANITY] |dE|<=70 positive ratio = {float((np.abs(y_de) <= 70.0).mean()):.4f}")

pair_idx = pd.DataFrame({
    "row": np.arange(len(df), dtype=int),
    "polar_id": polar_id,
    "npolar_id": npolar_id,
    "dE_meV": y_de,
})
if form_col is not None:
    pair_idx["formula"] = df[form_col].astype(str).values

pair_idx.to_csv(OUT_PAIR_INDEX, index=False)
print("[SAVED] pair index ->", OUT_PAIR_INDEX)


# ============================================================
# 6) CIF existence check for all unique IDs
# ============================================================
uniq_ids = pd.unique(np.concatenate([polar_id, npolar_id]))
print("\n==============================")
print("[FULL CACHE] unique structure IDs =", len(uniq_ids))

missing = []
t0 = time.time()
for k, sid in enumerate(uniq_ids, 1):
    try:
        _ = resolve_cif_path(sid)
    except Exception as e:
        missing.append((sid, str(e)))

    if k % 500 == 0:
        print(f"  checked {k}/{len(uniq_ids)} ... elapsed {time.time() - t0:.1f}s")

if missing:
    md = pd.DataFrame(missing, columns=["struct_id", "error"])
    md.to_csv(OUT_MISSING_CIF, index=False)
    print("[ERROR] missing CIFs =", len(missing))
    print(md.head(10))
    print("[SAVED] missing list ->", OUT_MISSING_CIF)
    raise FileNotFoundError(f"Missing CIF for {len(missing)} structure IDs. Fix CIF_DIR or IDs first.")
else:
    print("[CIF] all structure IDs resolved.")


# ============================================================
# 7) Build graph cache for all unique IDs
# ============================================================
stats = []
t0 = time.time()
built = 0

for k, sid in enumerate(uniq_ids, 1):
    fp = cache_file(sid)
    if os.path.exists(fp):
        g = joblib.load(fp)
    else:
        g = load_or_build_graph(sid)
        built += 1

    nat = int(g.get("natoms", len(g["z"])))
    E = int(g["edge_index"].shape[1])
    T = int(g.get("triplet_cos", np.zeros((0,), np.float32)).shape[0])
    mean_deg = float(E) / max(1, nat)

    stats.append((sid, nat, E, T, mean_deg, g.get("cif_path", "")))

    if k % 300 == 0:
        print(f"  cached {k}/{len(uniq_ids)} (newly built={built}) ... elapsed {time.time() - t0:.1f}s")

stats_df = pd.DataFrame(stats, columns=["struct_id", "natoms", "n_edges", "n_triplets", "mean_degree", "cif_path"])
stats_df.to_csv(OUT_GRAPH_STATS, index=False)

nat = stats_df["natoms"].values
E = stats_df["n_edges"].values
T = stats_df["n_triplets"].values

print("\n==============================")
print("[CACHE DONE]")
print("  CACHE_DIR =", CACHE_DIR)
print("  newly built =", built, "/", len(uniq_ids))
print("  natoms min/med/max =", int(nat.min()), int(np.median(nat)), int(nat.max()))
print("  edges  min/med/mean/p95/max =", int(E.min()), int(np.median(E)), float(E.mean()), int(np.quantile(E, 0.95)), int(E.max()))
print("  trip   min/med/mean/p95/max =", int(T.min()), int(np.median(T)), float(T.mean()), int(np.quantile(T, 0.95)), int(T.max()))
print("[SAVED] graph stats ->", OUT_GRAPH_STATS)


# ============================================================
# 8) Optional: build loaders from a split map
# ============================================================
if BUILD_LOADERS:
    assert SPLIT_MAP_PATH is not None and os.path.exists(SPLIT_MAP_PATH), "BUILD_LOADERS=True requires SPLIT_MAP_PATH"

    sm = pd.read_csv(SPLIT_MAP_PATH)
    if "row" not in sm.columns:
        raise KeyError(f"SPLIT_MAP must contain 'row'. Found: {list(sm.columns)}")

    idx_te = sm.loc[sm["split_outer"].astype(str).str.lower().eq("test"), "row"].values.astype(np.int64)
    idx_tr = sm.loc[sm["split_inner"].astype(str).str.lower().eq("train"), "row"].values.astype(np.int64)
    idx_va = sm.loc[sm["split_inner"].astype(str).str.lower().eq("val"), "row"].values.astype(np.int64)

    y_fe = (np.abs(y_de) <= 70.0).astype(np.int64)

    class PairGraphDataset(Dataset):
        def __init__(self, indices):
            self.indices = np.array(indices, dtype=np.int64)

        def __len__(self):
            return len(self.indices)

        def __getitem__(self, k):
            i = int(self.indices[k])
            gp = load_or_build_graph(polar_id[i])
            gn = load_or_build_graph(npolar_id[i])
            return i, gp, gn, float(y_de[i]), int(y_fe[i])

    def collate_pair(batch):
        idxs = torch.tensor([b[0] for b in batch], dtype=torch.long)
        y = torch.tensor([b[3] for b in batch], dtype=torch.float32)
        yfe = torch.tensor([b[4] for b in batch], dtype=torch.long)
        gp = [b[1] for b in batch]
        gn = [b[2] for b in batch]
        return idxs, gp, gn, y, yfe

    dl_tr = DataLoader(PairGraphDataset(idx_tr), batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_pair)
    dl_va = DataLoader(PairGraphDataset(idx_va), batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_pair)
    dl_te = DataLoader(PairGraphDataset(idx_te), batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_pair)

    print("\n[DATALOADER] ready:", len(dl_tr), len(dl_va), len(dl_te))
    idxs, gp_b, gn_b, y_b, yfe_b = next(iter(dl_tr))
    print("[BATCH] example polar =", gp_b[0]["id"], "E =", gp_b[0]["edge_index"].shape[1], "T =", gp_b[0]["triplet_cos"].shape[0])
    print("[DONE] loaders are in memory: dl_tr / dl_va / dl_te")

print("\n[DONE] full graph cache + stats ready. OUT_DIR =", OUT_DIR)

In [ ]:
# ============================================================
# CELL 1 — Build graph cache with angle triplets
#
# What this cell does:
#   1) Read the full pair database CSV
#   2) Parse polar / nonpolar structure IDs
#   3) Check CIF availability
#   4) Build periodic graphs with:
#        - radius cutoff
#        - per-center topK neighbors
#        - forced bidirectional edges
#        - angle triplets on primary edges
#   5) Save per-structure graph cache (.joblib)
#   6) Save graph statistics and pair index tables
#
# Optional:
#   If BUILD_LOADERS=True and SPLIT_MAP_PATH is provided, also build
#   dl_tr / dl_va / dl_te for quick inspection.
# ============================================================

import os
import re
import glob
import time
import math
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import torch
from torch.utils.data import Dataset, DataLoader
from ase.io import read as ase_read
from ase.neighborlist import neighbor_list


# ============================================================
# 0) User config
# ============================================================
DATA_PATH = "./data/pair_database.csv"
CIF_DIR = "./data/structures"

OUT_DIR = "./artifacts/graph_cache_angle"
CACHE_DIR = os.path.join(OUT_DIR, "graph_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# Optional split map for quick dataloader construction
SPLIT_MAP_PATH = None   # e.g. "./data/split_map.csv"
BUILD_LOADERS = False

OUT_GRAPH_STATS = os.path.join(OUT_DIR, "graph_stats.csv")
OUT_MISSING_CIF = os.path.join(OUT_DIR, "missing_cif_rows.csv")
OUT_PAIR_INDEX = os.path.join(OUT_DIR, "pair_index_from_db.csv")


# ============================================================
# 1) CSV column names
# ============================================================
POL_PAIRKEY_COL = "Polar_mpid"
NPOL_ID_COL = "NPolar_mpid"
DE_COL = "Energy_diff_meV"

FORM_COL_CANDIDATES = ["Polar_pretty_formula", "pretty_formula", "formula"]


# ============================================================
# 2) Graph config
# ============================================================
R_MAX = 6.0
MAX_NEIGHBORS = 64
SELF_LOOPS = False

K_ANGLE = 12
MAX_TRIPLETS_PER_CENTER = 200

# Optional loaders
BATCH_SIZE = 8
NUM_WORKERS = 0
PIN_MEMORY = True


# ============================================================
# 3) Helpers: CIF index + strict ID parsing
# ============================================================
def build_cif_index(cif_dir: str):
    files = glob.glob(os.path.join(cif_dir, "*.cif"))
    if len(files) == 0:
        files = glob.glob(os.path.join(cif_dir, "**", "*.cif"), recursive=True)

    idx = {}
    dup = 0
    for fp in files:
        key = os.path.splitext(os.path.basename(fp))[0].strip().lower()
        if key in idx:
            dup += 1
            continue
        idx[key] = fp

    print(f"[CIF] indexed: {len(idx)} files (raw={len(files)}, dups_skipped={dup}) under: {cif_dir}")
    return idx


CIF_INDEX = build_cif_index(CIF_DIR)


def normalize_token(s: str) -> str:
    s = str(s).strip()
    s = os.path.basename(s)
    if s.lower().endswith(".cif"):
        s = s[:-4]
    return s.strip()


_ID_TOKEN_RE = re.compile(r"(?i)^[a-z]+-\d+$|^\d+$")
_ID_EXTRACT_RE = re.compile(r"(?i)([a-z]+-\d+|\d+)")


def extract_id_token(s: str) -> str:
    s = normalize_token(s)
    m = _ID_EXTRACT_RE.findall(s)
    if not m:
        raise ValueError(f"Cannot extract ID token from: {s}")
    return m[0]


def assert_struct_id(s: str) -> str:
    s = normalize_token(s)
    if "__" in s:
        raise ValueError(f"[BUG] struct_id contains '__' (looks like pairkey): {s}")
    tok = extract_id_token(s)
    if not _ID_TOKEN_RE.match(tok):
        raise ValueError(f"Bad struct id token: {tok} from {s}")
    return tok


def parse_pairkey(pairkey: str, fallback_npolar: str = None):
    s = normalize_token(pairkey)
    if "__" in s:
        a, b = s.split("__", 1)
        return extract_id_token(a), extract_id_token(b)
    if fallback_npolar is not None:
        return extract_id_token(s), extract_id_token(fallback_npolar)
    raise ValueError(f"Cannot parse pairkey: {pairkey}")


def resolve_cif_path(struct_id: str) -> str:
    sid = assert_struct_id(struct_id)
    key = sid.lower()
    if key in CIF_INDEX:
        return CIF_INDEX[key]

    # Optional fuzzy fallback
    cands = [k for k in CIF_INDEX.keys() if key in k]
    if not cands:
        raise FileNotFoundError(f"Cannot find CIF for id={sid} under {CIF_DIR}")
    cands = sorted(cands, key=lambda x: (len(x), x))
    return CIF_INDEX[cands[0]]


def cache_file(struct_id: str) -> str:
    sid = assert_struct_id(struct_id)
    return os.path.join(CACHE_DIR, f"{sid}.joblib")


def _unique_rows_int64(mat: np.ndarray):
    v = np.ascontiguousarray(mat).view(np.dtype((np.void, mat.dtype.itemsize * mat.shape[1])))
    _, first = np.unique(v, return_index=True)
    first = np.sort(first)
    return first


# ============================================================
# 4) Graph build: V2 + ANGLE
# ============================================================
def build_pbc_graph_v2_angle_with_remap(atoms, r_max: float, max_neighbors: int, self_loops: bool, k_angle: int):
    pos = atoms.get_positions().astype(np.float32)
    z = atoms.get_atomic_numbers().astype(np.int64)
    cell = atoms.get_cell().array.astype(np.float32)
    pbc = np.array(atoms.get_pbc(), dtype=np.bool_)

    i, j, S, _d = neighbor_list("ijSd", atoms, cutoff=r_max, self_interaction=self_loops)

    if len(i) == 0:
        return dict(
            z=z,
            pos=pos,
            cell=cell,
            pbc=pbc,
            edge_index=np.zeros((2, 0), dtype=np.int64),
            edge_vec=np.zeros((0, 3), dtype=np.float32),
            edge_dist=np.zeros((0,), dtype=np.float32),
            edge_shift=np.zeros((0, 3), dtype=np.float32),
            triplet_center=np.zeros((0,), dtype=np.int64),
            triplet_e1=np.zeros((0,), dtype=np.int64),
            triplet_e2=np.zeros((0,), dtype=np.int64),
            triplet_cos=np.zeros((0,), dtype=np.float32),
        )

    i = i.astype(np.int64)
    j = j.astype(np.int64)
    S = S.astype(np.int64)

    shift = (S.astype(np.float32) @ cell).astype(np.float32)
    rij = (pos[j] + shift - pos[i]).astype(np.float32)
    dist = np.linalg.norm(rij, axis=1).astype(np.float32)

    # Primary edges: per-center topK
    if max_neighbors is not None and max_neighbors > 0:
        order = np.lexsort((dist, i))
        i2, j2, rij2, dist2, S2 = i[order], j[order], rij[order], dist[order], S[order]

        keep = np.zeros(len(i2), dtype=bool)
        start = 0
        while start < len(i2):
            ii = i2[start]
            end = start + 1
            while end < len(i2) and i2[end] == ii:
                end += 1
            k = min(int(max_neighbors), end - start)
            keep[start:start + k] = True
            start = end

        i2, j2, rij2, dist2, S2 = i2[keep], j2[keep], rij2[keep], dist2[keep], S2[keep]
    else:
        i2, j2, rij2, dist2, S2 = i, j, rij, dist, S

    E0 = int(len(i2))

    # Angle triplets on primary edges
    trip_c, trip_e1, trip_e2, trip_cos = [], [], [], []
    if E0 >= 2:
        ord_i = np.argsort(i2, kind="mergesort")
        i_sorted = i2[ord_i]
        start = 0
        while start < E0:
            c = int(i_sorted[start])
            end = start + 1
            while end < E0 and int(i_sorted[end]) == c:
                end += 1

            eidx = ord_i[start:end]
            m = len(eidx)
            if m >= 2:
                if m > k_angle:
                    sel = np.argsort(dist2[eidx], kind="mergesort")[:k_angle]
                    eidx = eidx[sel]
                    m = len(eidx)

                if m >= 2:
                    U = rij2[eidx] / (dist2[eidx][:, None] + 1e-12)
                    cosmat = (U @ U.T).astype(np.float32)
                    t0, t1 = np.triu_indices(m, k=1)

                    if t0.size > 0:
                        cosv = cosmat[t0, t1]
                        e1 = eidx[t0].astype(np.int64)
                        e2 = eidx[t1].astype(np.int64)

                        if cosv.size > MAX_TRIPLETS_PER_CENTER:
                            cosv = cosv[:MAX_TRIPLETS_PER_CENTER]
                            e1 = e1[:MAX_TRIPLETS_PER_CENTER]
                            e2 = e2[:MAX_TRIPLETS_PER_CENTER]

                        trip_c.append(np.full((len(cosv),), c, dtype=np.int64))
                        trip_e1.append(e1)
                        trip_e2.append(e2)
                        trip_cos.append(cosv.astype(np.float32))

            start = end

    if len(trip_c) == 0:
        triplet_center0 = np.zeros((0,), dtype=np.int64)
        triplet_e10 = np.zeros((0,), dtype=np.int64)
        triplet_e20 = np.zeros((0,), dtype=np.int64)
        triplet_cos0 = np.zeros((0,), dtype=np.float32)
    else:
        triplet_center0 = np.concatenate(trip_c, axis=0)
        triplet_e10 = np.concatenate(trip_e1, axis=0)
        triplet_e20 = np.concatenate(trip_e2, axis=0)
        triplet_cos0 = np.concatenate(trip_cos, axis=0)

    # Force bidirectional edges
    i_all0 = np.concatenate([i2, j2], axis=0).astype(np.int64)
    j_all0 = np.concatenate([j2, i2], axis=0).astype(np.int64)
    S_all0 = np.concatenate([S2, -S2], axis=0).astype(np.int64)
    rij_all0 = np.concatenate([rij2, -rij2], axis=0).astype(np.float32)
    dist_all0 = np.concatenate([dist2, dist2], axis=0).astype(np.float32)

    # Deduplicate by (i, j, S)
    key0 = np.stack([i_all0, j_all0, S_all0[:, 0], S_all0[:, 1], S_all0[:, 2]], axis=1).astype(np.int64)
    first = _unique_rows_int64(key0)

    new_index = -np.ones((len(i_all0),), dtype=np.int64)
    new_index[first] = np.arange(len(first), dtype=np.int64)

    # Remap triplets
    if triplet_e10.size > 0:
        ne1 = new_index[triplet_e10]
        ne2 = new_index[triplet_e20]
        ok = (ne1 >= 0) & (ne2 >= 0)
        triplet_center = triplet_center0[ok]
        triplet_e1 = ne1[ok].astype(np.int64)
        triplet_e2 = ne2[ok].astype(np.int64)
        triplet_cos = triplet_cos0[ok].astype(np.float32)
    else:
        triplet_center = triplet_center0
        triplet_e1 = triplet_e10
        triplet_e2 = triplet_e20
        triplet_cos = triplet_cos0

    i_all = i_all0[first]
    j_all = j_all0[first]
    S_all = S_all0[first]
    rij_all = rij_all0[first]
    dist_all = dist_all0[first]
    shift_all = (S_all.astype(np.float32) @ cell).astype(np.float32)

    edge_index = np.stack([i_all, j_all], axis=0)

    return dict(
        z=z,
        pos=pos,
        cell=cell,
        pbc=pbc,
        edge_index=edge_index,
        edge_vec=rij_all,
        edge_dist=dist_all,
        edge_shift=shift_all,
        triplet_center=triplet_center,
        triplet_e1=triplet_e1,
        triplet_e2=triplet_e2,
        triplet_cos=triplet_cos,
    )


def load_or_build_graph(struct_id: str):
    sid = assert_struct_id(struct_id)
    cpath = cache_file(sid)
    if os.path.exists(cpath):
        return joblib.load(cpath)

    cif = resolve_cif_path(sid)
    atoms = ase_read(cif)
    g = build_pbc_graph_v2_angle_with_remap(
        atoms,
        r_max=R_MAX,
        max_neighbors=MAX_NEIGHBORS,
        self_loops=SELF_LOOPS,
        k_angle=K_ANGLE,
    )
    g["id"] = sid
    g["cif_path"] = cif
    g["natoms"] = int(len(g["z"]))
    joblib.dump(g, cpath)
    return g


# ============================================================
# 5) Read full database CSV + build ID arrays
# ============================================================
df = pd.read_csv(DATA_PATH)
print(f"[CSV] rows={len(df)} cols={len(df.columns)}")

for c in [POL_PAIRKEY_COL, NPOL_ID_COL, DE_COL]:
    if c not in df.columns:
        raise KeyError(f"Missing column '{c}' in DATA_PATH. Found: {list(df.columns)}")

y_de = pd.to_numeric(df[DE_COL], errors="coerce").values.astype(np.float32)
if not np.isfinite(y_de).all():
    bad = np.where(~np.isfinite(y_de))[0][:20]
    raise ValueError(f"dE has NaN/Inf rows={bad}")

form_col = next((c for c in FORM_COL_CANDIDATES if c in df.columns), None)

polar_id = np.empty(len(df), dtype=object)
npolar_id = np.empty(len(df), dtype=object)
mismatch = 0

for i, (pk, ncol) in enumerate(zip(df[POL_PAIRKEY_COL].values, df[NPOL_ID_COL].values)):
    p, n_from_pk = parse_pairkey(pk, fallback_npolar=ncol)
    n = extract_id_token(ncol)
    polar_id[i] = assert_struct_id(p)
    npolar_id[i] = assert_struct_id(n)
    if normalize_token(n_from_pk) != normalize_token(n):
        mismatch += 1

print(f"[SANITY] npolar mismatch between pairkey and {NPOL_ID_COL} = {mismatch}/{len(df)}")
print(f"[SANITY] |dE|<=70 positive ratio = {float((np.abs(y_de) <= 70.0).mean()):.4f}")

pair_idx = pd.DataFrame({
    "row": np.arange(len(df), dtype=int),
    "polar_id": polar_id,
    "npolar_id": npolar_id,
    "dE_meV": y_de,
})
if form_col is not None:
    pair_idx["formula"] = df[form_col].astype(str).values

pair_idx.to_csv(OUT_PAIR_INDEX, index=False)
print("[SAVED] pair index ->", OUT_PAIR_INDEX)


# ============================================================
# 6) CIF existence check for all unique IDs
# ============================================================
uniq_ids = pd.unique(np.concatenate([polar_id, npolar_id]))
print("\n==============================")
print("[FULL CACHE] unique structure IDs =", len(uniq_ids))

missing = []
t0 = time.time()
for k, sid in enumerate(uniq_ids, 1):
    try:
        _ = resolve_cif_path(sid)
    except Exception as e:
        missing.append((sid, str(e)))

    if k % 500 == 0:
        print(f"  checked {k}/{len(uniq_ids)} ... elapsed {time.time() - t0:.1f}s")

if missing:
    md = pd.DataFrame(missing, columns=["struct_id", "error"])
    md.to_csv(OUT_MISSING_CIF, index=False)
    print("[ERROR] missing CIFs =", len(missing))
    print(md.head(10))
    print("[SAVED] missing list ->", OUT_MISSING_CIF)
    raise FileNotFoundError(f"Missing CIF for {len(missing)} structure IDs. Fix CIF_DIR or IDs first.")
else:
    print("[CIF] all structure IDs resolved.")


# ============================================================
# 7) Build graph cache for all unique IDs
# ============================================================
stats = []
t0 = time.time()
built = 0

for k, sid in enumerate(uniq_ids, 1):
    fp = cache_file(sid)
    if os.path.exists(fp):
        g = joblib.load(fp)
    else:
        g = load_or_build_graph(sid)
        built += 1

    nat = int(g.get("natoms", len(g["z"])))
    E = int(g["edge_index"].shape[1])
    T = int(g.get("triplet_cos", np.zeros((0,), np.float32)).shape[0])
    mean_deg = float(E) / max(1, nat)

    stats.append((sid, nat, E, T, mean_deg, g.get("cif_path", "")))

    if k % 300 == 0:
        print(f"  cached {k}/{len(uniq_ids)} (newly built={built}) ... elapsed {time.time() - t0:.1f}s")

stats_df = pd.DataFrame(stats, columns=["struct_id", "natoms", "n_edges", "n_triplets", "mean_degree", "cif_path"])
stats_df.to_csv(OUT_GRAPH_STATS, index=False)

nat = stats_df["natoms"].values
E = stats_df["n_edges"].values
T = stats_df["n_triplets"].values

print("\n==============================")
print("[CACHE DONE]")
print("  CACHE_DIR =", CACHE_DIR)
print("  newly built =", built, "/", len(uniq_ids))
print("  natoms min/med/max =", int(nat.min()), int(np.median(nat)), int(nat.max()))
print("  edges  min/med/mean/p95/max =", int(E.min()), int(np.median(E)), float(E.mean()), int(np.quantile(E, 0.95)), int(E.max()))
print("  trip   min/med/mean/p95/max =", int(T.min()), int(np.median(T)), float(T.mean()), int(np.quantile(T, 0.95)), int(T.max()))
print("[SAVED] graph stats ->", OUT_GRAPH_STATS)


# ============================================================
# 8) Optional: build loaders from a split map
# ============================================================
if BUILD_LOADERS:
    assert SPLIT_MAP_PATH is not None and os.path.exists(SPLIT_MAP_PATH), "BUILD_LOADERS=True requires SPLIT_MAP_PATH"

    sm = pd.read_csv(SPLIT_MAP_PATH)
    if "row" not in sm.columns:
        raise KeyError(f"SPLIT_MAP must contain 'row'. Found: {list(sm.columns)}")

    idx_te = sm.loc[sm["split_outer"].astype(str).str.lower().eq("test"), "row"].values.astype(np.int64)
    idx_tr = sm.loc[sm["split_inner"].astype(str).str.lower().eq("train"), "row"].values.astype(np.int64)
    idx_va = sm.loc[sm["split_inner"].astype(str).str.lower().eq("val"), "row"].values.astype(np.int64)

    y_fe = (np.abs(y_de) <= 70.0).astype(np.int64)

    class PairGraphDataset(Dataset):
        def __init__(self, indices):
            self.indices = np.array(indices, dtype=np.int64)

        def __len__(self):
            return len(self.indices)

        def __getitem__(self, k):
            i = int(self.indices[k])
            gp = load_or_build_graph(polar_id[i])
            gn = load_or_build_graph(npolar_id[i])
            return i, gp, gn, float(y_de[i]), int(y_fe[i])

    def collate_pair(batch):
        idxs = torch.tensor([b[0] for b in batch], dtype=torch.long)
        y = torch.tensor([b[3] for b in batch], dtype=torch.float32)
        yfe = torch.tensor([b[4] for b in batch], dtype=torch.long)
        gp = [b[1] for b in batch]
        gn = [b[2] for b in batch]
        return idxs, gp, gn, y, yfe

    dl_tr = DataLoader(PairGraphDataset(idx_tr), batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_pair)
    dl_va = DataLoader(PairGraphDataset(idx_va), batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_pair)
    dl_te = DataLoader(PairGraphDataset(idx_te), batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_pair)

    print("\n[DATALOADER] ready:", len(dl_tr), len(dl_va), len(dl_te))
    idxs, gp_b, gn_b, y_b, yfe_b = next(iter(dl_tr))
    print("[BATCH] example polar =", gp_b[0]["id"], "E =", gp_b[0]["edge_index"].shape[1], "T =", gp_b[0]["triplet_cos"].shape[0])
    print("[DONE] loaders are in memory: dl_tr / dl_va / dl_te")

print("\n[DONE] full graph cache + stats ready. OUT_DIR =", OUT_DIR)